# LangChain을 활용한 RAG 실습

LangChain은 LLM에 입력되는 프롬프트에 다양한 엔지니어링 요소(프롬프트 템플릿, 체인, 메모리, 툴 호출 등)를 결합하여 하나의 완성된 애플리케이션 형태로 구성할 수 있는 단계별 워크플로우 프레임워크입니다.

또한 OpenAI, Hugging Face 등 주요 LLM 플랫폼과 손쉽게 연동할 수 있어 확장성이 뛰어납니다.

### 필요 라이브러리 설치

In [ ]:
!pip install langchain langchain-core langchain_community faiss-cpu tiktoken langchain-openai langchain-text-splitters datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.3/84.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 1.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


## LangChain 프롬프트 엔지니어링

LangChain은 입력된 프롬프트를 탬플릿화 시키고 불러온 LLM 모델과 연결하여 단계별 워크플로우를 하나의 프로세스로 합쳐 줍니다.  



**단계별 워크플로우**:

    1. LLM 모델로드
    2. 프롬프트 템플릿 생성
    3. LLM 모델과 템플릿을 연결(Chains)
    4. 사용자 입력을 받아 모델에 입력
    5. 모델의 결과를 후처리

In [ ]:
# 2. 모듈 임포트
# getpass: 터미널에서 사용자 입력을 안전하게 받기 위한 모듈 (API 키 입력 시 표시되지 않음)
from getpass import getpass
# ChatOpenAI: LangChain에서 OpenAI의 채팅 모델을 사용하기 위한 클래스
from langchain_openai import ChatOpenAI
# os: 운영체제와 상호작용하기 위한 모듈 (환경 변수 설정에 사용)
import os

# 3. OpenAI API 키 설정
# 사용자로부터 API 키를 입력받습니다 (비밀번호처럼 표시되지 않음)
# 입력받은 키는 openai_api_key 변수에 문자열로 저장됩니다
openai_api_key = getpass("Enter your OpenAI API key: ")

# 4. 환경 변수 설정
# LangChain과 OpenAI 라이브러리는 기본적으로 환경 변수에서 API 키를 읽어옵니다
# os.environ 딕셔너리에 키를 설정하면 이후 모든 OpenAI 관련 호출에서 이 키가 자동으로 사용됩니다
# 이 방식은 코드에 API 키를 하드코딩하는 것보다 보안상 안전합니다
os.environ["OPENAI_API_KEY"] = openai_api_key

Enter your OpenAI API key: ··········


In [ ]:
# 5. LLM 모델 초기화
# ChatOpenAI 클래스의 인스턴스를 생성하여 GPT-4o-mini 모델을 로드합니다
# model="gpt-4o-mini": 사용할 모델의 식별자 (OpenAI에서 제공하는 모델 이름)
# temperature=0.0: 생성의 무작위성 조절 (0.0은 가장 결정적이고 일관된 응답 생성)
# 이 llm 객체는 이후 LangChain 체인에서 실제 텍스트 생성을 담당하는 컴포넌트가 됩니다
# 인스턴스 생성 시 내부적으로 OpenAI API에 연결하고, 사용 가능한 모델 목록을 확인합니다
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)

# 코드 실행 완료 시점:
# 1. 모든 필요한 패키지가 설치되고 임포트됨
# 2. API 키가 환경 변수에 설정됨
# 3. llm 객체가 초기화되어 텍스트 생성 요청을 보낼 준비가 됨
# 이제 이 llm 객체를 활용하여 LangChain의 다양한 기능(프롬프트 템플릿, 체인, 메모리 등)을 구성할 수 있습니다

### 프롬프트 템플릿 생성

`langchain_core.prompts` 모듈은 `PromptTemplate` 객체를 제공하여 사용자가 입력한 프롬프트에 적절한 템플릿을 덮어 씌워 모델에 입력할수 있게 도와줍니다.  
이때 `PromptTemplate`에 입력되는 문자열 안에 `{변수이름}`형식으로 수정 가능한 변수를 할당할 수 있습니다.

In [ ]:
# langchain_core.prompts 모듈에서 PromptTemplate 클래스를 임포트합니다
# PromptTemplate: 재사용 가능한 프롬프트 템플릿을 생성하는 클래스로,
# 고정된 형식에 변수를 삽입할 수 있는 구조를 제공합니다
from langchain_core.prompts import PromptTemplate

# 텍스트 요약 템플릿
# summary_template 변수에 문자열 형태의 프롬프트 템플릿을 정의합니다
# 이 템플릿은 LLM이 수행할 작업(전문 요약가 역할)과 구체적 지시사항(50단어 이내 요약)을 포함합니다
# {text_to_summarize}: 플레이스홀더(자리 표시자)로, 실제 요약할 텍스트가 삽입될 위치를 표시합니다
# 중괄호 {} 안의 변수명은 나중에 실제 값으로 대체됩니다
summary_template = """
당신은 전문 요약가입니다.
아래 주어진 텍스트를 50단어 이내로 간결하게 요약해 주세요.

텍스트:
{text_to_summarize}

요약:
"""

# PromptTemplate 객체 생성
# PromptTemplate.from_template() 클래스 메서드를 사용하여 템플릿 문자열에서 PromptTemplate 인스턴스를 생성합니다
# 이 메서드는 템플릿 문자열을 분석하여 {text_to_summarize} 변수를 인식하고,
# 해당 변수를 입력 변수로 갖는 템플릿 객체를 생성합니다
# 생성된 summary_prompt 객체는 텍스트 요약 작업을 위한 표준화된 프롬프트 형식이 됩니다
# 이 객체는 나중에 사용자 입력(실제 요약할 텍스트)을 받아 완성된 프롬프트로 조합할 수 있습니다
summary_prompt = PromptTemplate.from_template(summary_template)

# 생성된 summary_prompt 객체의 특징:
# 1. 입력 변수: ['text_to_summarize'] (하나의 입력 변수를 가짐)
# 2. 템플릿 형식: 위에 정의한 문자열 그대로 저장됨
# 3. 용도: 동일한 형식으로 반복적인 요약 작업 수행 시 재사용 가능

# 이 템플릿 객체를 사용하면:
# 1. summary_prompt.format(text_to_summarize="실제 텍스트") 방식으로 완성된 프롬프트 생성 가능
# 2. LangChain 체인(chain)에서 다른 컴포넌트와 연결하여 자동화된 워크플로우 구성 가능
# 3. 일관된 프롬프트 형식을 유지하며 다양한 텍스트에 적용 가능

# PromptTemplate의 장점:
# - 코드와 프롬프트 내용 분리: 프롬프트 수정이 코드 변경 없이 가능
# - 변수 관리: 동일한 템플릿에 다른 값을 쉽게 삽입 가능
# - 재사용성: 동일 작업에 대해 여러 번 재사용 가능
# - 유지보수성: 중앙에서 템플릿 관리 가능

### LLM - Prompt 체인 만들기 (LCEL 방식)

LangChain에서는 기존의 LLMChain.run() 방식 대신,
**LangChain Expression Language(LCEL)**을 사용해
프롬프트 → LLM → 출력 파싱 과정을 하나의 체인으로 간단히 구성할 수 있습니다.

summary_prompt | llm | StrOutputParser() 형태로 체인을 만들고,
.invoke()에 사용자 입력을 넣으면 최종 생성 텍스트가 반환됩니다.

In [ ]:
# langchain_core.output_parsers 모듈에서 StrOutputParser 클래스를 임포트합니다
# StrOutputParser: LLM의 출력을 문자열로 변환하는 파서로, 모델 응답에서 텍스트만 추출합니다
from langchain_core.output_parsers import StrOutputParser

# LangChain Expression Language (LCEL)를 사용하여 체인 생성
# LCEL은 LangChain에서 컴포넌트를 연결하는 선언적 방식으로, 파이프라인 구축을 간소화합니다
# '|' 연산자를 사용하여 컴포넌트들을 연결합니다 (Unix 파이프라인과 유사한 개념)
# summary_chain은 세 개의 컴포넌트가 순차적으로 연결된 파이프라인입니다:
# 1. summary_prompt: PromptTemplate 객체 (템플릿에 변수 삽입)
# 2. llm: ChatOpenAI 객체 (실제 LLM 호출)
# 3. StrOutputParser(): 출력 파서 (LLM 응답을 문자열로 변환)
# 이 체인은 입력 → 프롬프트 완성 → LLM 전달 → 응답 파싱의 전체 과정을 캡슐화합니다
summary_chain = summary_prompt | llm | StrOutputParser()

# 사용자 프롬프트 (요약할 원본 텍스트)
# text_to_summarize 변수에 요약할 장문의 텍스트를 문자열로 저장합니다
# 이 텍스트는 뉴스 기사로, 앱 마켓 실태조사에 관한 내용을 포함하고 있습니다
# 실제 응용에서는 데이터베이스, 파일, 웹 스크래핑 등 다양한 소스에서 텍스트를 가져올 수 있습니다
text_to_summarize = """
[데일리안 = 조인영 기자] 우리나라 앱 개발자가 앱 마켓사업자에게서 경험하는 주요 불공정 사례는 심사 지연과 등록 거부이며, 앱 내 결제(인앱결제)의 가장 큰 문제점은 ‘과도한 수수료’라고 인식하고 있는 것으로 나타났다.

방송통신위원회와 한국인터넷진흥원은 11일 '전기통신사업법' 제22조의9(앱 마켓사업자의 의무 및 실태조사)에 따른 ‘2024년도 앱 마켓 실태조사 결과’를 발표했다.

2023년도 국내 ‘앱 마켓’ 규모는 거래액 기준으로 8조1952억원으로 2022년 8조 7598억원 대비 6.4% 감소했다

각 사업자별로 보면, 애플 앱스토어(10.1%)와 삼성전자 갤럭시스토어(6.3%)는 전년 대비 매출이 증가했으며, 구글 플레이(△10.1%)와 원스토어(△21.6%)는 감소했다.

4개 앱 마켓사업자의 거래액 대비 수수료 비중은 약 14~26% 수준이며, 앱 등록 매출의 경우 애플 앱스토어는 약 9.4% 증가했고 구글 플레이는 약 12.9% 감소했다.

또한 국내 앱 마켓에 등록된 전체 앱 수는 전년 대비 0.1% 증가한 531만8182개(각 앱 마켓별 중복 포함), 앱 개발자 수는 전년 대비 0.65% 적은 163만6655명(각 앱 마켓별 중복 포함)으로 나타났다.

분야별 앱 등록 비중은 사진‧도구(26.1%), 라이프스타일(15.6%), 미디어‧출판(14.5%) 관련 앱이 전체의 56.2%를 차지했다.

국내 앱 개발자가 이용하는 앱 마켓은 구글 플레이(96.4%), 애플 앱스토어(71.3%) 순(중복포함)이며, 매출액 비중도 구글 플레이가 67.5%로 가장 높았다. 그 를 애플 앱스토어(28.2%), 원스토어(2.9%), 갤럭시스토어(1.5%)가 따랐다.

앱 개발자가 느끼는 주요 불공정 사례로는 앱 심사지연 경험(애플 앱스토어 6.8%, 구글 플레이 26.2%)이 가장 높았으며, 앱 등록 거부 경험(애플 20%, 구글 3%)과 앱 삭제 경험(구글 8.2%, 애플 3.2%)이 그 뒤를 이었다.

앱 내 결제의 가장 큰 문제점은 ‘과도한 수수료’라고 답한 앱 개발자가 0.4%로 가장 높게 나타났으며, 다음으로 ‘환불 등 수익 정산의 불명확함’(11.6%), ‘결제 수단 선택 제한’(8.9%) 순으로 조사됐다.

앱 개발자가 느낀 앱 최초 등록 과정상의 어려움으로는 ‘심사 기준이 확하지 않음’(구글 플레이 29.8%, 애플 앱스토어 29.6%), ‘질의에 대한 드백 지연’(각각 26.1%, 25.3%) 등이 꼽혔다.

앱을 최초로 등록하기 위해 소요되는 심사기간은 구글 플레이는 등록 시 일 이내(25.6%)에 처리되는 경우가 많았으나, 애플 앱스토어는 6∼7일 이내(42.5%)로 나타났다.

우리나라 국민이 가장 자주 이용하는 앱 마켓은 구글 플레이(67.2%), 애플 스토어(29.7%) 순이었으며, 해당 앱 마켓을 자주 이용하는 이유로는 ‘사용이 편리해서’(67.7%), ‘설치되어 있어서’(61.3%), ‘상품 수가 많아서’(33,5%) 등 으로 나타났다.
"""

# 사용자 프롬프트를 체인에 입력
# summary_chain.invoke() 메서드를 호출하여 체인을 실행합니다
# 딕셔너리 형식으로 입력 변수(text_to_summarize)에 실제 텍스트를 매핑합니다
# invoke() 메서드는 내부적으로 다음 순서로 처리합니다:
# 1. summary_prompt에 {text_to_summarize} 변수를 실제 텍스트로 대체하여 완성된 프롬프트 생성
# 2. 완성된 프롬프트를 llm 모델에 전달하여 API 호출
# 3. LLM의 응답을 StrOutputParser()로 전달하여 문자열 추출
# 4. 최종 문자열 결과를 summary_result 변수에 반환
summary_result = summary_chain.invoke({"text_to_summarize": text_to_summarize})

# 요약 결과 출력
# print 함수를 사용하여 콘솔에 결과를 표시합니다
print("[요약 결과]")
print(summary_result)

# 전체 실행 흐름 요약:
# 1. 텍스트 입력 → 2. 프롬프트 템플릿에 삽입 → 3. LLM에 전달 → 4. 응답 생성 → 5. 출력 파싱 → 6. 결과 출력
# 이 체인은 재사용 가능하며, 다른 텍스트에 대해 동일한 요약 작업을 반복할 수 있습니다

# LCEL 체인의 장점:
# - 가독성: 선언적 구문으로 데이터 흐름을 명확히 표현
# - 유연성: 컴포넌트 추가/제거가 쉬움
# - 재사용성: 동일 체인을 여러 입력에 적용 가능
# - 디버깅 용이성: 각 단계별 출력 확인 가능

[요약 결과]
국내 앱 개발자들은 앱 마켓에서 심사 지연과 등록 거부를 주요 불공정 사례로 경험하며, 인앱 결제의 과도한 수수료를 문제로 인식하고 있다. 2023년 앱 마켓 규모는 감소했으며, 구글 플레이와 애플 앱스토어가 가장 많이 사용된다.


### 롤 토큰 Chat 템플릿

In [ ]:
# langchain_core.prompts 모듈에서 ChatPromptTemplate 클래스를 임포트합니다
# ChatPromptTemplate: 대화형 프롬프트를 위한 템플릿으로, 시스템 메시지와 사용자 메시지를 구조화합니다
from langchain_core.prompts import ChatPromptTemplate
# langchain_core.output_parsers 모듈에서 StrOutputParser 클래스를 임포트합니다
# StrOutputParser: LLM의 출력을 문자열로 변환하는 파서입니다
from langchain_core.output_parsers import StrOutputParser

# ChatPromptTemplate 생성: 시스템/사용자 메시지 정의
# ChatPromptTemplate.from_messages() 메서드를 사용하여 메시지 목록으로 템플릿을 생성합니다
# 메시지 목록은 튜플의 리스트 형식으로, 각 튜플은 (역할, 내용) 구조를 가집니다
# 여기서는 두 개의 메시지를 정의합니다:
# 1. 시스템 메시지: "당신은 질문의 상황에 적절한 인공지능 모델을 찾아주는 AI 전문가입니다"
#    - LLM의 역할과 행동 방향성을 설정하는 지시사항입니다
# 2. 사용자 메시지: "다음 상황에 맞는 모델을 찾아줘 \n{input}"
#    - 사용자의 질문 템플릿으로, {input} 변수를 포함하여 실제 상황 설명이 삽입될 위치를 표시합니다
# ChatPromptTemplate은 대화 히스토리를 관리하거나 멀티턴 대화를 지원하는 데 유용합니다
chat_template = ChatPromptTemplate.from_messages([
    ("system", "당신은 질문의 상황에 적절한 인공지능 모델을 찾아주는 AI 전문가입니다"),
    ("user", "다음 상황에 맞는 모델을 찾아줘 \n{input}")
])

# LCEL(LangChain Expression Language)을 이용한 체인 구성
# '|' 연산자를 사용하여 컴포넌트들을 순차적으로 연결합니다:
# 1. chat_template: ChatPromptTemplate 객체 (시스템/사용자 메시지 템플릿)
# 2. llm: 이전에 초기화한 ChatOpenAI 객체 (GPT-4o-mini 모델)
# 3. StrOutputParser(): LLM 응답을 문자열로 변환하는 출력 파서
# chat_chain은 입력 → 프롬프트 구성 → LLM 호출 → 응답 파싱의 전체 과정을 캡슐화합니다
chat_chain = chat_template | llm | StrOutputParser()

# 체인 실행
# chat_chain.invoke() 메서드를 호출하여 체인을 실행합니다
# 딕셔너리 형식으로 입력 변수(input)에 실제 상황 설명을 매핑합니다:
# {input: "우리는 CCTV에 범죄 상황을 인지하기 위한 기능을 넣으려고 한다"}
# invoke() 메서드의 내부 처리 흐름:
# 1. chat_template에 {input} 변수를 실제 텍스트로 대체하여 완성된 대화 프롬프트 생성
#    - 시스템 메시지와 사용자 메시지가 포함된 구조화된 프롬프트가 생성됨
# 2. 완성된 프롬프트를 llm 모델에 전달하여 OpenAI API 호출
#    - LLM은 시스템 지시사항을 고려하여 사용자의 상황에 맞는 AI 모델을 추천하는 응답 생성
# 3. LLM의 응답을 StrOutputParser()로 전달하여 문자열로 변환
# 4. 최종 문자열 결과를 response 변수에 반환
response = chat_chain.invoke({"input": "우리는 CCTV에 범죄 상황을 인지하기 위한 기능을 넣으려고 한다"})

# 응답 출력
# print 함수를 사용하여 응답을 콘솔에 출력합니다
# "[응답]" 레이블을 추가하여 가독성을 높입니다
print("[응답]\n", response)

# 이 코드의 특징:
# - ChatPromptTemplate을 사용하여 역할 기반 대화 프롬프트를 구성했습니다
# - LCEL의 선언적 구문으로 간결하게 체인을 구성했습니다
# - 체인 실행 시 딕셔너리로 입력 변수를 전달하여 유연성을 확보했습니다

# ChatPromptTemplate의 장점:
# - 시스템 메시지와 사용자 메시지를 명확히 분리하여 역할을 부여할 수 있음
# - 대화 히스토리 관리가 용이함 (여러 턴의 대화를 지원)
# - 다양한 메시지 타입(system, human, ai, function 등)을 지원

# 실행 결과 예상:
# LLM은 CCTV 범죄 상황 인지 기능에 적합한 AI 모델(예: 객체 감지 모델, 행동 인식 모델 등)을 추천하는 응답을 생성할 것입니다
# 시스템 메시지에 정의된 역할에 따라 전문가처럼 상황에 맞는 모델을 설명할 것입니다

[응답]
 CCTV에 범죄 상황을 인지하기 위한 기능을 추가하려면, 다음과 같은 인공지능 모델을 고려할 수 있습니다:

1. **객체 탐지 모델**: YOLO (You Only Look Once), SSD (Single Shot MultiBox Detector), Faster R-CNN과 같은 모델을 사용하여 CCTV 영상에서 사람, 차량, 가방 등 특정 객체를 실시간으로 탐지할 수 있습니다.

2. **행동 인식 모델**: LSTM (Long Short-Term Memory) 또는 3D CNN과 같은 모델을 활용하여 사람의 행동을 분석하고, 비정상적인 행동(예: 싸움, 도난 등)을 인식할 수 있습니다.

3. **이상 탐지 모델**: Autoencoder, Isolation Forest와 같은 비지도 학습 모델을 사용하여 정상적인 행동 패턴을 학습하고, 이를 기반으로 이상 행동을 탐지할 수 있습니다.

4. **비디오 분석 플랫폼**: OpenVINO, TensorFlow Object Detection API와 같은 플랫폼을 활용하여 다양한 모델을 통합하고, 실시간으로 CCTV 영상을 분석할 수 있습니다.

이러한 모델들을 조합하여 범죄 상황을 효과적으로 인지하고 경고할 수 있는 시스템을 구축할 수 있습니다.


### 메모리 활용

LLM은 **이전 대화 내용(히스토리)**을 함께 넣어주면 대화를 더 자연스럽게 이어갈 수 있습니다.

LangChain 최신 버전에서는
history라는 자리에 누적된 대화 메시지를 자동으로 넣어주는 기능을 제공합니다.
그래서 같은 세션 ID로 체인을 호출하면, 이전 질문·답변들이 자동으로 기억되고 → 다음 질문에 더 자연스러운 답변을 하게 됩니다.

In [ ]:
# langchain_openai 모듈에서 ChatOpenAI 클래스를 임포트합니다
# ChatOpenAI: OpenAI의 채팅 모델을 LangChain에서 사용하기 위한 클래스입니다
from langchain_openai import ChatOpenAI

# langchain_core.prompts 모듈에서 ChatPromptTemplate과 MessagesPlaceholder를 임포트합니다
# ChatPromptTemplate: 대화형 프롬프트 템플릿을 생성하는 클래스
# MessagesPlaceholder: 대화 기록(히스토리)이 삽입될 자리를 표시하는 특별한 플레이스홀더
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# langchain_core.runnables.history 모듈에서 RunnableWithMessageHistory를 임포트합니다
# RunnableWithMessageHistory: 대화 기록을 관리하면서 실행 가능한 체인을 감싸는 래퍼 클래스
from langchain_core.runnables.history import RunnableWithMessageHistory

# langchain_core.chat_history 모듈에서 InMemoryChatMessageHistory를 임포트합니다
# InMemoryChatMessageHistory: 메모리에 대화 기록을 저장하는 클래스 (휘발성, 세션별 관리)
from langchain_core.chat_history import InMemoryChatMessageHistory

# 1) LLM 모델 초기화
# ChatOpenAI 클래스의 인스턴스를 생성하여 GPT-4o-mini 모델을 로드합니다
# model="gpt-4o-mini": 사용할 모델 식별자
# temperature 매개변수가 명시되지 않았으므로 기본값(0.7)이 사용됩니다
# 이 llm 객체는 이전에 생성한 llm 객체와 별개로 새로 생성됩니다 (동일 모델 사용)
llm = ChatOpenAI(model="gpt-4o-mini")

# 2) history + user_input 프롬프트 템플릿 생성
# ChatPromptTemplate.from_messages() 메서드를 사용하여 메시지 목록으로 템플릿을 생성합니다
# 이 템플릿은 세 부분으로 구성됩니다:
# 1. 시스템 메시지: "너는 친절한 AI 비서야."
#    - LLM의 역할과 행동 스타일을 정의합니다 (친절한 비서 역할)
# 2. MessagesPlaceholder(variable_name="history"): 대화 기록이 삽입될 자리
#    - variable_name="history"로 지정되어, 나중에 'history' 변수에 실제 대화 기록이 채워집니다
#    - 이 플레이스홀더는 여러 개의 이전 메시지(사용자 질문, AI 응답)를 포함할 수 있습니다
# 3. 사용자 메시지: ("human", "{user_input}")
#    - 현재 사용자의 새 입력이 들어갈 자리, {user_input} 변수는 실행 시 실제 사용자 입력으로 대체됩니다
# 이 구조는 매번 대화할 때: [시스템 메시지] + [과거 대화 기록] + [현재 사용자 입력] 형식으로 프롬프트가 구성됨을 의미합니다
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 친절한 AI 비서야."),  # 시스템 역할 정의
        MessagesPlaceholder(variable_name="history"),  # 이전 대화 기록이 채워질 자리
        ("human", "{user_input}"),  # 현재 사용자 입력이 들어갈 자리
    ]
)

# 이 코드까지의 상태:
# 1. 새로운 llm 객체가 생성되었습니다 (기본 temperature 설정)
# 2. prompt 템플릿이 정의되었지만 아직 실제 사용되지 않았습니다
# 3. MessagesPlaceholder를 통해 대화 기록을 관리할 수 있는 구조가 준비되었습니다

# 다음 단계에서 할 일 (이 코드에는 없지만 일반적인 흐름):
# 1. 이 prompt와 llm을 연결하여 기본 체인 생성
# 2. RunnableWithMessageHistory로 래핑하여 대화 기록 관리 기능 추가
# 3. 세션별로 대화 기록을 저장할 저장소(store) 설정
# 4. 실제 대화 실행

# MessagesPlaceholder의 작동 원리:
# - 대화 기록은 LangChain의 Message 객체 리스트 형식으로 저장됩니다
# - 각 Message는 'type'(human, ai, system 등)과 'content'를 가집니다
# - MessagesPlaceholder는 이 리스트를 자동으로 적절한 형식의 프롬프트로 변환합니다

# InMemoryChatMessageHistory의 특징:
# - 메모리에 대화 기록을 저장하므로 프로그램 재시작 시 기록이 사라집니다
# - 세션별로 별도의 기록을 관리할 수 있습니다
# - 대화 기록이 너무 길어지면 토큰 제한을 초과할 수 있어, 필요시 자르거나 요약하는 로직이 추가될 수 있습니다

# 이 구조의 장점:
# - 대화의 맥락을 유지하면서 자연스러운 대화 가능
# - 사용자가 이전에 말한 내용을 참조할 수 있음
# - AI의 응답도 일관성 있게 유지 가능

In [ ]:
# 3) 기본 체인 생성: 프롬프트 → LLM 연결
# '|' 연산자를 사용하여 LCEL(LangChain Expression Language) 방식으로 체인을 구성합니다
# base_chain은 두 개의 컴포넌트가 연결된 간단한 체인입니다:
# 1. prompt: 이전에 정의한 ChatPromptTemplate 객체 (시스템 메시지 + 히스토리 플레이스홀더 + 사용자 입력)
# 2. llm: ChatOpenAI 객체 (GPT-4o-mini 모델)
# 이 base_chain은 아직 대화 기록 관리 기능이 없는 기본 체인입니다
# 실행 시: prompt가 입력을 받아 완성된 프롬프트를 생성 → llm이 이 프롬프트를 처리하여 응답 생성
base_chain = prompt | llm

# 4) 세션별 메모리(대화 기록) 저장소 설정
# store 변수는 빈 딕셔너리로 초기화됩니다
# 이 딕셔너리는 세션 ID를 키로, 해당 세션의 대화 기록 객체(InMemoryChatMessageHistory)를 값으로 저장합니다
# 예: {"session-1": InMemoryChatMessageHistory 객체, "session-2": 다른 InMemoryChatMessageHistory 객체}
# 각 세션은 독립적인 대화 기록을 유지합니다 (예: 다른 사용자, 다른 채팅방)
store = {}

# get_session_history 함수 정의: 세션별 대화 기록 관리자
# 이 함수는 세션 ID를 입력받아 해당 세션의 대화 기록 객체를 반환합니다
# session_id: str - 세션을 구분하는 고유 식별자 (예: "user-123", "chat-room-1")
# 반환 타입: InMemoryChatMessageHistory - 대화 기록을 저장하는 객체
def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    # store 딕셔너리에 해당 session_id 키가 존재하지 않으면 (새로운 세션인 경우)
    if session_id not in store:
        # 새로운 InMemoryChatMessageHistory 객체를 생성하여 store에 저장합니다
        # InMemoryChatMessageHistory는 메모리 내에서 대화 메시지(Message 객체들)를 리스트 형태로 관리합니다
        store[session_id] = InMemoryChatMessageHistory()
    # 해당 세션의 대화 기록 객체를 반환합니다 (기존 것이거나 새로 생성된 것)
    return store[session_id]

# 5) RunnableWithMessageHistory로 "메모리 있는 체인" 만들기
# RunnableWithMessageHistory 클래스는 기본 체인에 대화 기록 관리 기능을 추가하는 래퍼(wrapper)입니다
# 이 클래스의 인스턴스를 생성하면 대화 기록이 자동으로 관리되는 체인이 만들어집니다
llm_chain_with_history = RunnableWithMessageHistory(
    # 첫 번째 인자: base_chain - 대화 기록 기능을 추가할 기본 체인
    # 이 체인은 prompt와 llm이 연결된 상태입니다
    base_chain,
    # 두 번째 인자: get_session_history - 세션별 대화 기록을 가져오는 함수
    # 이 함수는 세션 ID를 받아 해당 세션의 대화 기록 객체를 반환합니다
    # RunnableWithMessageHistory는 내부적으로 이 함수를 호출하여 대화 기록을 관리합니다
    get_session_history,
    # 세 번째 인자: input_messages_key="user_input"
    # 입력 딕셔너리에서 어떤 키의 값을 사용자 메시지로 간주할지 지정합니다
    # 체인 실행 시 invoke()에 전달하는 딕셔너리에서 "user_input" 키 아래의 값이 현재 사용자 입력으로 처리됩니다
    # 이 값은 대화 기록에 "human" 타입 메시지로 저장되고, 프롬프트의 {user_input} 변수에도 전달됩니다
    input_messages_key="user_input",
    # 네 번째 인자: history_messages_key="history"
    # 프롬프트 템플릿에서 정의한 MessagesPlaceholder의 변수 이름을 지정합니다
    # 이 이름은 prompt = ChatPromptTemplate.from_messages([...])에서 정의한 MessagesPlaceholder의 variable_name과 일치해야 합니다
    # RunnableWithMessageHistory는 이 키를 사용하여 대화 기록을 프롬프트의 적절한 위치에 삽입합니다
    history_messages_key="history",
)

# RunnableWithMessageHistory의 내부 작동 원리:
# 1. 사용자가 체인을 실행할 때 session_id를 포함한 설정(config)을 제공합니다
# 2. RunnableWithMessageHistory는 get_session_history(session_id)를 호출하여 해당 세션의 대화 기록을 가져옵니다
# 3. 현재 사용자 입력(input_messages_key로 지정된 값)을 대화 기록에 "human" 메시지로 임시 추가합니다
# 4. 전체 대화 기록(이전 대화 + 현재 사용자 입력)을 history_messages_key로 지정된 위치에 삽입하여 프롬프트를 완성합니다
# 5. 완성된 프롬프트를 base_chain에 전달하여 LLM 응답을 생성합니다
# 6. LLM의 응답을 대화 기록에 "ai" 메시지로 추가합니다 (사용자 입력도 영구적으로 기록에 추가됨)
# 7. 응답을 사용자에게 반환합니다

# 이제 llm_chain_with_history는 대화 기록을 유지하며 작동하는 체인이 되었습니다
# 같은 session_id로 체인을 여러 번 호출하면 이전 대화 내용이 자동으로 히스토리에 포함됩니다

# 세션 관리의 장점:
# - 여러 사용자나 채팅방을 별도로 관리할 수 있습니다
# - 각 세션은 독립적인 대화 기록을 유지합니다
# - 메모리 저장소(store)를 필요에 따라 데이터베이스나 파일 저장으로 교체할 수 있습니다

# InMemoryChatMessageHistory의 제한사항:
# - 메모리에 저장되므로 프로그램 재시작 시 모든 대화 기록이 손실됩니다
# - 대량의 세션을 관리할 때는 메모리 사용량이 증가할 수 있습니다
# - 실제 프로덕션 환경에서는 Redis, 데이터베이스 등의 영구 저장소를 사용하는 커스텀 구현이 필요할 수 있습니다

# 다음 단계에서는 이 체인을 실제로 실행하여 대화 기록이 어떻게 관리되는지 확인할 것입니다

In [ ]:
# 6) 체인 실행 (첫 대화)
# llm_chain_with_history 체인의 invoke() 메서드를 호출하여 첫 번째 대화를 시작합니다
# invoke() 메서드는 두 개의 인자를 받습니다:
# 1. 입력 데이터: {"user_input": "BERT 모델에 대하여 간략한 설명 부탁해"}
#    - user_input 키에는 사용자의 실제 질문이 들어갑니다
#    - 이 값은 프롬프트 템플릿의 {user_input} 변수에 삽입되고, 동시에 대화 기록에 "human" 타입 메시지로 저장됩니다
# 2. 설정(config): {"configurable": {"session_id": "session-1"}}
#    - configurable 딕셔너리 안에 session_id를 지정하여 "session-1"이라는 세션을 식별합니다
#    - 같은 session_id를 사용하는 모든 호출은 동일한 대화 기록을 공유하게 됩니다
#    - 이 설정은 RunnableWithMessageHistory가 get_session_history("session-1")를 호출하도록 합니다
response1 = llm_chain_with_history.invoke(
    {"user_input": "BERT 모델에 대하여 간략한 설명 부탁해"},
    config={"configurable": {"session_id": "session-1"}},  # 세션 ID 지정 - "session-1" 세션의 대화 기록 사용/관리
)

# 응답 출력
# response1은 LLM의 응답을 담고 있는 객체로, content 속성에 실제 응답 텍스트가 들어 있습니다
# print 함수를 사용하여 응답 내용을 콘솔에 출력합니다
print("[응답 1]\n", response1.content)

# 이 코드의 실행 시 내부에서 일어나는 일들:
# 1. 세션 기록 조회: get_session_history("session-1") 함수가 호출되어 "session-1" 세션의 대화 기록을 가져옵니다
#    - 처음 호출이므로 store 딕셔너리에 "session-1" 키가 없어 새로운 InMemoryChatMessageHistory 객체가 생성됩니다
#    - 이 객체는 빈 대화 기록으로 시작합니다 (아직 메시지가 없음)
# 2. 사용자 입력 기록: 사용자의 입력("BERT 모델에 대하여 간략한 설명 부탁해")이 대화 기록에 "human" 타입 메시지로 추가됩니다
# 3. 프롬프트 구성: ChatPromptTemplate이 다음 세 부분으로 프롬프트를 구성합니다
#    a) 시스템 메시지: "너는 친절한 AI 비서야."
#    b) 대화 기록: 현재는 빈 기록 (첫 대화이므로)
#    c) 현재 사용자 입력: "BERT 모델에 대하여 간략한 설명 부탁해"
# 4. LLM 호출: 완성된 프롬프트가 llm 모델(GPT-4o-mini)로 전달되어 응답이 생성됩니다
# 5. 응답 기록: LLM의 응답이 대화 기록에 "ai" 타입 메시지로 추가됩니다
#    - 이제 "session-1" 세션의 대화 기록에는 [human: 질문, ai: 응답] 두 메시지가 저장됩니다
# 6. 응답 반환: 생성된 응답이 response1 객체로 반환됩니다

# response1 객체의 구조:
# - content: LLM이 생성한 실제 응답 텍스트 (예: "BERT는 Bidirectional Encoder Representations from Transformers의 약자로...")
# - additional_kwargs: 기타 메타데이터 (예: 토큰 사용량, 완료 이유 등)
# - response_metadata: 응답에 대한 추가 정보

# 세션 관리의 중요성:
# - session_id를 통해 여러 대화를 독립적으로 관리할 수 있습니다
# - 예를 들어, "session-1"은 한 사용자, "session-2"는 다른 사용자의 대화를 별도로 유지합니다
# - 웹 애플리케이션에서 각 사용자마다 고유한 session_id를 부여하면 사용자별 대화 기록을 관리할 수 있습니다

# 이 실행 이후의 상태:
# - store 딕셔너리에는 "session-1" 키가 생성되고, 그 값으로 InMemoryChatMessageHistory 객체가 저장됩니다
# - 이 객체에는 첫 번째 질문과 응답이 순서대로 저장되어 있습니다
# - 다음에 같은 session_id로 체인을 호출하면 이 기록이 자동으로 포함됩니다

# 출력 결과 예상:
# "[응답 1]"이라는 헤더 아래에 BERT 모델에 대한 간략한 설명이 출력됩니다
# LLM은 시스템 메시지("친절한 AI 비서")의 지시에 따라 친절한 어조로 설명할 것입니다

[응답 1]
 BERT(Bidirectional Encoder Representations from Transformers)는 Google이 2018년에 발표한 자연어 처리(NLP) 모델로, 텍스트의 맥락을 이해하기 위해 양방향 Transformer 아키텍처를 사용합니다. BERT는 단어의 의미를 문맥에 따라 파악할 수 있도록 훈련되며, 특히 문맥에서 단어의 의미가 어떻게 변하는지를 반영합니다.

BERT의 주요 특징은 다음과 같습니다:

1. **양방향성**: BERT는 입력 텍스트를 왼쪽과 오른쪽 문맥을 모두 고려하여 처리하므로, 각 단어의 의미를 보다 정확하게 이해할 수 있습니다.

2. **사전 훈련 및 전이 학습**: BERT는 대량의 텍스트 데이터로 사전 훈련된 후, 특정 NLP 작업에 맞게 미세 조정(fine-tuning)될 수 있습니다. 이 접근법은 다양한 자연어 처리 과제를 효과적으로 처리할 수 있는 모델을 만드는 데 유용합니다.

3. **Masked Language Model**: BERT는 일부 단어를 마스킹하고, 모델이 이를 예측하도록 훈련됩니다. 이 과정은 모델이 문맥을 기반으로 단어를 이해하도록 도와줍니다.

4. **다양한 NLP 작업에 적용 가능**: BERT는 질의 응답, 감정 분석, 문서 요약 등 다양한 자연어 처리 작업에서 뛰어난 성능을 보여줍니다.

BERT는 자연어 처리 분야에 큰 영향을 미쳤으며, 이후 다른 많은 모델들이 BERT의 아이디어를 기반으로 발전되었습니다.


In [ ]:
# 7) 체인 실행 (두 번째 대화) - 이전 대화가 history에 자동 저장/사용됨
# llm_chain_with_history 체인을 다시 호출하여 두 번째 대화를 진행합니다
# 이번에도 동일한 session_id인 "session-1"을 사용하여 이전 대화 기록을 계속 유지합니다
response2 = llm_chain_with_history.invoke(
    # 입력 데이터: {"user_input": "방금 설명한 모델과 GPT 모델의 차이점은?"}
    # - 사용자가 BERT와 GPT 모델의 차이점을 묻는 두 번째 질문입니다
    # - 이 값은 프롬프트의 {user_input} 변수에 삽입되고, 대화 기록에 새로운 "human" 메시지로 추가됩니다
    {"user_input": "방금 설명한 모델과 GPT 모델의 차이점은?"},
    # 설정(config): {"configurable": {"session_id": "session-1"}}
    # - 첫 번째 대화와 동일한 "session-1" 세션 ID를 지정합니다
    # - 이로 인해 get_session_history("session-1")이 호출되고, 이전 대화 기록이 포함된 동일한 InMemoryChatMessageHistory 객체가 반환됩니다
    config={"configurable": {"session_id": "session-1"}},
)

# 응답 출력
# 첫 번째 응답과 구분하기 위해 개행 문자(\n)와 "[응답 2]" 헤더를 추가하여 출력합니다
print("\n[응답 2]\n", response2.content)

# 두 번째 대화 실행 시 내부에서 일어나는 일들:
# 1. 세션 기록 조회: get_session_history("session-1") 함수가 호출되어 "session-1" 세션의 대화 기록을 가져옵니다
#    - 이전 대화에서 이미 생성된 InMemoryChatMessageHistory 객체가 반환됩니다
#    - 이 객체에는 첫 번째 대화의 두 메시지가 저장되어 있습니다: [human: BERT 설명 요청, ai: BERT 설명]
# 2. 사용자 입력 기록: 두 번째 질문("방금 설명한 모델과 GPT 모델의 차이점은?")이 대화 기록에 새로운 "human" 타입 메시지로 추가됩니다
#    - 이제 대화 기록에는 세 개의 메시지가 저장됩니다: [첫 질문, 첫 응답, 두 번째 질문]
# 3. 프롬프트 구성: ChatPromptTemplate이 다음 세 부분으로 프롬프트를 구성합니다
#    a) 시스템 메시지: "너는 친절한 AI 비서야."
#    b) 대화 기록: 첫 번째 질문과 응답이 포함된 전체 대화 기록 (3개 메시지)
#    c) 현재 사용자 입력: "방금 설명한 모델과 GPT 모델의 차이점은?"
# 4. LLM 호출: 완성된 프롬프트가 llm 모델로 전달됩니다. 이 프롬프트에는 이전 대화의 맥락이 포함되어 있으므로 LLM은 "방금 설명한 모델"이 BERT를 가리킨다는 것을 이해합니다
# 5. 응답 기록: LLM의 응답이 대화 기록에 새로운 "ai" 타입 메시지로 추가됩니다
#    - 이제 "session-1" 세션의 대화 기록에는 네 개의 메시지가 저장됩니다: [질문1, 응답1, 질문2, 응답2]
# 6. 응답 반환: 생성된 응답이 response2 객체로 반환됩니다

# 대화 기록의 자동 관리:
# - RunnableWithMessageHistory가 대화 기록을 자동으로 관리하므로, 개발자는 명시적으로 기록을 저장하거나 불러올 필요가 없습니다
# - 같은 session_id를 사용하는 한, 대화 기록은 계속해서 누적됩니다

# 두 번째 질문에서의 맥락 이해:
# - 사용자가 "방금 설명한 모델"이라고 했을 때, LLM은 대화 기록을 참조하여 이전에 BERT에 대해 설명했음을 알 수 있습니다
# - 따라서 LLM은 BERT와 GPT의 차이점에 초점을 맞춘 응답을 생성합니다
# - 대화 기록이 없었다면 LLM은 "방금 설명한 모델"이 어떤 모델인지 이해하지 못했을 것입니다

# 출력 결과 예상:
# - LLM은 BERT와 GPT 모델의 주요 차이점(예: BERT는 양방향 인코더, GPT는 단방향 디코더; 사전 학습 목적의 차이 등)을 설명할 것입니다
# - 시스템 메시지의 "친절한 AI 비서" 역할에 따라 쉽고 친절한 설명이 제공될 것입니다

# 이 코드가 보여주는 LangChain의 강점:
# 1. 대화 기록의 자동 관리: 개발자가 직접 기록을 관리할 필요 없이 세션 기반 대화를 쉽게 구현할 수 있습니다
# 2. 맥락 보존: 이전 대화 내용을 자동으로 참조하여 일관된 대화를 유지할 수 있습니다
# 3. 확장성: 다른 저장소(데이터베이스, 파일 시스템 등)로 대화 기록을 저장하도록 쉽게 변경할 수 있습니다

# 세션 관리의 실제 적용 예:
# - 웹 애플리케이션에서 사용자별로 고유한 session_id(예: 사용자 ID)를 부여하면 사용자마다 개인화된 대화 기록을 유지할 수 있습니다
# - 채팅봇 서비스에서 각 채팅방마다 다른 session_id를 사용하여 독립적인 대화를 관리할 수 있습니다

# 메모리 사용 고려사항:
# - 대화 기록이 계속 누적되면 프롬프트가 너무 길어져 토큰 제한을 초과할 수 있습니다
# - 실제 애플리케이션에서는 오래된 메시지를 제거하거나 요약하는 기능을 추가해야 할 수 있습니다
# - LangChain에서는 대화 기록을 자르거나 요약하는 다양한 메모리 관리 전략을 제공합니다

# 이제 대화 기록이 있는 체인의 기본 작동 원리를 이해했습니다.
# 첫 번째 대화와 두 번째 대화가 동일한 세션 ID를 공유함으로써 자연스러운 대화 흐름이 유지됩니다.


[응답 2]
 BERT 모델과 GPT(Generative Pre-trained Transformer) 모델은 모두 Transformer 아키텍처를 기반으로 한 자연어 처리 모델이지만, 몇 가지 중요한 차이점이 있습니다.

### 1. **목적과 용도**
- **BERT**: 주로 자연어 이해(NLU) 작업에 중점을 두고 있으며, 문맥을 이해하고 텍스트를 분석하는 데 강점이 있습니다. 예를 들어, 질문 응답, 감정 분석, 문장 분류 등의 작업에 적합합니다.
- **GPT**: 텍스트 생성(generative) 모델로, 주로 자연어 생성(NLG) 작업에 사용됩니다. 주어진 텍스트에 기반하여 다음 단어나 문장을 생성하는 데 적합합니다. 예를 들어, 대화 생성, 이야기 작성, 문서 요약 등의 작업에 활용됩니다.

### 2. **훈련 방식**
- **BERT**: 양방향으로 문맥을 이해하기 위해 "Masked Language Model"과 "Next Sentence Prediction"이라는 두 가지 태스크로 사전 훈련됩니다. 입력 문장 내에서 일부 단어를 마스킹하고, 모델이 그 단어를 예측하도록 합니다.
- **GPT**: 단방향(왼쪽에서 오른쪽)으로 훈련됩니다. 주어진 단어 시퀀스에서 다음 단어를 예측하는 방식으로 훈련됩니다. 이로 인해 GPT는 자연어 생성에 더 적합합니다.

### 3. **아키텍처**
- **BERT**: Transformer의 인코더(encoder) 부분만 사용합니다. 양방향 처리를 통해 텍스트의 전반적인 맥락을 파악합니다.
- **GPT**: Transformer의 디코더(decoder) 부분만 사용します. 데이터의 흐름이 왼쪽에서 오른쪽으로 진행되며, 이전 단어들만 참고하여 다음 단어를 생성합니다.

### 4. **작업 적용 차이**
- **BERT**: 특정 작업에 맞게 미세 조정(fine-tuning)하여 사용하는 경우가 많습니다.
- **GPT**: 미세 조정 없이도 프롬프트(prompt)를 통해 다양한 작업을 수행할 수 있으며